In [27]:
import pandas as pd
import re
from difflib import get_close_matches


In [28]:
# Read the genetic data
genetic_data = pd.read_csv(r"C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1\GeneMarkers-95 varieties 1.csv")

genetic_data.info
genetic_data.head(10)

,taglo_id,ATLANTIC_124776,ESCORT_159228,ARGOS_120931,ALTURAS_329540,ELMUNDO_835520,NADINE_241950,VIOLETQUEEN_3345402,DEODARA_513721,MEMPHIS_2279529,...,DIAMANT_153502,INNOVATOR_234757,TRIPLE7_3347283,SPUNTA_283648,ANTI_118190,CARRERA_142554,FORTUS_3279866,SMART_1883446,SALINERO_253609,FABULA_171173
0,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,10,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,13,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,17,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,21,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,22,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,39,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,48,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,49,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,54,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [29]:
genetic_data.isnull().sum()

taglo_id           0
ATLANTIC_124776    0
ESCORT_159228      0
ARGOS_120931       0
ALTURAS_329540     0
                  ..
CARRERA_142554     0
FORTUS_3279866     0
SMART_1883446      0
SALINERO_253609    0
FABULA_171173      0
Length: 95, dtype: int64

In [30]:
# Clean column names
genetic_columns_clean = [c.strip().upper().replace(" ", "").replace(".", "") for c in genetic_data.columns]

# Assign cleaned names back to DataFrame
genetic_data.columns = genetic_columns_clean

# Now you can view columns
print(genetic_data.columns)


Index(['TAGLO_ID', 'ATLANTIC_124776', 'ESCORT_159228', 'ARGOS_120931',
       'ALTURAS_329540', 'ELMUNDO_835520', 'NADINE_241950',
       'VIOLETQUEEN_3345402', 'DEODARA_513721', 'MEMPHIS_2279529',
       'AGRIA_125336', 'PAYETTERUSSET_4617676', 'IVORYRUSSET_1360551',
       'SABABA_4959615', 'CECILE_147140', 'FRISIA_163592', 'CARDYMA_3345873',
       'RICKEYRUSSET_4640884', 'ANIVIA_4446779', 'AVARNA_1630052',
       'DONALD_158741', 'JAERLA_231308', 'ARIZONA_3056017',
       'SARPOMIRA_1629377', 'MARILYN_1632173', 'DESIREE_139717',
       'MUSE_6159099', 'ADORA_123802', 'TAURUS_1883495', 'BERBER_121293',
       'RANGERRUSSET_265488', 'BINTJE_126680', 'FENWAYRED_4055844',
       'KONDOR_248484', 'SAGITTA_1459411', 'CLEARWATERR_2721777',
       'NICOLA_248062', 'TETONRUSSET_3029162', 'FESTIEN_172619',
       'HANSA_180216', 'PEEWEERUSSET_3347291', 'PRINCEOFORANGE_416115',
       'SUPERIOR_1477892', 'VANGOGH_294181', 'CHALLENGER_1883438',
       'JENNIFER_3221728', 'PARELLA_2983716', 'CH

In [31]:
merged_aroma_sensory= pd.read_csv(r'C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1\merged_aroma_sensory.csv')
print(merged_aroma_sensory)

         Variety  1-Nonanol (Floral, waxy)  \
0          ADORA                       0.0   
1          AGRIA                       0.0   
2       ALOUETTE                       0.0   
3         ALTHEA                       0.0   
4        ALTURAS                       0.0   
..           ...                       ...   
89  TETON RUSSET                       0.0   
90       TRIPLE7                       0.0   
91      VAN GOGH                       0.0   
92     VE 71-105                       0.0   
93  VIOLET QUEEN                       0.0   

    2,3-Butanedione (Butter, creamy, sweet)  \
0                                       0.0   
1                                       0.0   
2                                       0.0   
3                                       0.0   
4                                       0.0   
..                                      ...   
89                                      0.0   
90                                      0.0   
91                      

In [32]:
import re

def normalize_variety(name):
    if not isinstance(name, str):
        return ""
    name = name.upper()
    name = name.replace(".", "")
    name = re.sub(r'\bR\b', '', name)
    name = re.sub(r'RUSSET', '', name)
    name = name.replace(" ", "")
    name = name.strip()
    name = name.split('_')[0]
    return name

# Normalize genetic data index
genetic_data_t = genetic_data.set_index('TAGLO_ID').T
genetic_data_t.index = genetic_data_t.index.map(normalize_variety)

# Same corrections dictionary
name_corrections = {
    'CLEARWATERR': 'CLEARWATER',
    'ALVERSTONER': 'ALVERSTONE'
}

# Apply corrections to genetic data index as well
genetic_data_t.index = genetic_data_t.index.to_series().replace(name_corrections)

# Normalize aroma sensory data
merged_aroma_sensory['Variety_norm'] = merged_aroma_sensory['Variety'].apply(normalize_variety)
merged_aroma_sensory['Variety_norm'] = merged_aroma_sensory['Variety_norm'].replace(name_corrections)
merged_aroma_sensory = merged_aroma_sensory.set_index('Variety_norm')

# Now merge on corrected index
combined_df = merged_aroma_sensory.merge(genetic_data_t, left_index=True, right_index=True, how='inner')

print(f"Combined shape after correction: {combined_df.shape}")

# Check missing varieties again
aroma_varieties = set(merged_aroma_sensory.index)
genetic_varieties = set(genetic_data_t.index)

missing_in_genetic = aroma_varieties - genetic_varieties
missing_in_aroma = genetic_varieties - aroma_varieties

print(f"Varieties missing in genetic data: {missing_in_genetic}")
print(f"Varieties missing in aroma/sensory data: {missing_in_aroma}")
combined_df.head(5)

Combined shape after correction: (94, 262461)
Varieties missing in genetic data: set()
Varieties missing in aroma/sensory data: set()


,Variety,"1-Nonanol (Floral, waxy)","2,3-Butanedione (Butter, creamy, sweet)","Benzaldehyde (Almond, cherry, fruity)","Butanal, 3-methyl- (Fruity, malty, chocolate)","Cyclotrisiloxane, hexamethyl- (Chemical, silicone-like)","Decanal (Citrus, floral)","Dimethyl trisulfide (Garlic, onion, sulfurous)","Furfural (Sweet, almond, caramel)","Hexanal (Green, grassy)",...,564850,564851,564852,564853,564854,564857,564858,564859,564860,564861
ADORA,ADORA,0.0,0.0,17.984870,15.836435,17.417208,18.203338,0.00000,0.0,0.00000,...,2,4,3,1,0,0,0,0,0,2
AGRIA,AGRIA,0.0,0.0,14.859493,14.154847,15.947810,14.499810,17.21077,0.0,0.00000,...,0,4,1,0,0,0,0,0,0,0
ALOUETTE,ALOUETTE,0.0,0.0,15.205705,15.262178,14.678352,17.462371,0.00000,0.0,0.00000,...,3,4,3,2,0,0,0,0,0,3
ALTHEA,ALTHEA,0.0,0.0,19.535081,14.984406,17.704453,21.158234,0.00000,0.0,0.00000,...,0,4,0,0,0,0,0,0,0,0
ALTURAS,ALTURAS,0.0,0.0,20.293776,14.482897,14.257052,16.590508,0.00000,0.0,17.24572,...,1,4,2,1,0,0,0,0,1,1


# Feature Reduction on X (Genetic Markers)

Variance thresholding (remove near-constant features)

In [ ]:
flavor_cols = [
    'Sweet', 'Intensity of Flavour', 'Metallic Flavour', 'Bitter Flavour',
    'Earthy Flavour', 'Sour Flavour', 'Fresh Flavour', 'Sweet Flavour',
    'Root/ Vegetable Flavour', 'Farmyard (grass/hay) flavour',
    'Bitter Aftertaste', 'Sour Aftertaste', 'Sweet Aftertaste'
]
exclude_cols = flavor_cols + ['Variety'] + [ 
    '1-Nonanol (Floral, waxy)', '2,3-Butanedione (Butter, creamy, sweet)', 'Benzaldehyde (Almond, cherry, fruity)',
    'Butanal, 3-methyl- (Fruity, malty, chocolate)', 'Cyclotrisiloxane, hexamethyl- (Chemical, silicone-like)',
    'Decanal (Citrus, floral)', 'Dimethyl trisulfide (Garlic, onion, sulfurous)', 'Furfural (Sweet, almond, caramel)',
    'Hexanal (Green, grassy)', 'Hexanoic acid (Fatty, cheesy, sweaty)', 'Methional (Cooked potato, sulfurous)', 
    'Octanal (Citrus, fruity)', 'Pentanal (Green, fatty, pungent)'
]

X = combined_df.drop(columns=exclude_cols, errors='ignore')
y = combined_df[flavor_cols]

print(f"X shape: {X.shape}, y shape: {y.shape}")


X shape: (94, 262434), y shape: (94, 13)


In [37]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)  # adjust threshold
X_reduced = selector.fit_transform(X)

print(f"Reduced X shape: {X_reduced.shape}")


Reduced X shape: (94, 261110)


Use feature selection based on correlation with target (univariate filter)

In [38]:
from sklearn.feature_selection import SelectKBest, f_regression

# For each flavor, select top features, or just once for general
selector = SelectKBest(score_func=f_regression, k=500)  # top 500 features

X_selected = selector.fit_transform(X, y['Sweet'].fillna(0))  # example flavor to select features
print(f"Shape after SelectKBest: {X_selected.shape}")


Shape after SelectKBest: (94, 500)


<!-- Lasso regression -->

In [39]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score, KFold
import numpy as np

# flavor columns
flavor_cols = [
    'Sweet', 'Intensity of Flavour', 'Metallic Flavour', 'Bitter Flavour',
    'Earthy Flavour', 'Sour Flavour', 'Fresh Flavour', 'Sweet Flavour',
    'Root/ Vegetable Flavour', 'Farmyard (grass/hay) flavour',
    'Bitter Aftertaste', 'Sour Aftertaste', 'Sweet Aftertaste'
]

for flavor in flavor_cols:
    y_flavor = y[flavor].dropna()
    mask = y[flavor].notna()
    
    X_train = X_selected[mask.values]
    y_train = y_flavor.values
    
    pls = PLSRegression(n_components=5)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pls, X_train, y_train, cv=cv, scoring='r2')
    
    print(f"{flavor} PLS CV R^2: {np.mean(scores):.3f} ± {np.std(scores):.3f}")


Sweet PLS CV R^2: 0.382 ± 0.375
Intensity of Flavour PLS CV R^2: 0.301 ± 0.228
Metallic Flavour PLS CV R^2: 0.063 ± 0.255
Bitter Flavour PLS CV R^2: 0.075 ± 0.251
Earthy Flavour PLS CV R^2: 0.247 ± 0.197
Sour Flavour PLS CV R^2: 0.131 ± 0.284
Fresh Flavour PLS CV R^2: 0.355 ± 0.200
Sweet Flavour PLS CV R^2: 0.339 ± 0.577
Root/ Vegetable Flavour PLS CV R^2: 0.303 ± 0.225
Farmyard (grass/hay) flavour PLS CV R^2: 0.201 ± 0.186
Bitter Aftertaste PLS CV R^2: 0.047 ± 0.260
Sour Aftertaste PLS CV R^2: 0.189 ± 0.301
Sweet Aftertaste PLS CV R^2: 0.151 ± 0.762


# Run Elastic Net & Random Forest on reduced dataset

In [48]:
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
import numpy as np

models = {
    'ElasticNet': ElasticNet(max_iter=5000, random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42)
}

for flavor in flavor_cols:
    mask = y[flavor].notna()
    X_train = X_selected.loc[mask]
    y_train = y.loc[mask, flavor]

    print(f"\nResults for {flavor}:")
    for name, model in models.items():
        scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
        print(f"{name} CV R^2: {np.mean(scores):.3f} ± {np.std(scores):.3f}")


AttributeError: 'numpy.ndarray' object has no attribute 'loc'

<!-- Filter genetic markers by variance -->